# Chapter 2: First-Order Ordinary Differential Equations

## Classification of Differential Equations

### Numerical and Symbolic Approaches in Computational Software

#### Symbolic Solution using SymPy

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

# Define symbolic variables and functions
t = sp.Symbol('t')
y = sp.Function('y')(t)

# Define the differential equation: dy/dt = 3t - 2y
eq = sp.Eq(y.diff(t), 3*t - 2*y)

# Solve the differential equation with initial condition y(0) = 4
solution = sp.dsolve(eq, y, ics={y.subs(t, 0): 4})
print("Symbolic solution:", solution)

# Convert symbolic solution to a numerical function
y_solution = sp.lambdify(t, solution.rhs, 'numpy')

# Plot the analytical solution
t_points = np.linspace(0, 3, 100)
y_points = y_solution(t_points)

plt.figure(figsize=(10, 6))
plt.plot(t_points, y_points, 'b-', label='Symbolic Solution')
plt.grid(True)
plt.xlabel('t')
plt.ylabel('y(t)')
plt.title('Symbolic Solution to dy/dt = 3t - 2y with y(0) = 4')
plt.legend()
plt.show()

#### Numerical Solution using SciPy

In [ ]:
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt

# Define the ODE system: dy/dt = 3t - 2y
def differential_eq(t, y):
    return 3*t - 2*y

# Initial condition and time span
y0 = [4]  # Initial value y(0) = 4
t_span = (0, 3)

# Solve using different numerical methods
methods = ['RK45', 'Radau', 'BDF']
plt.figure(figsize=(12, 7))

for method in methods:
    # Solve the IVP with the current method
    sol = solve_ivp(differential_eq, t_span, y0, method=method,
                   dense_output=True, rtol=1e-6, atol=1e-9)

    # Generate smooth output for plotting
    t_eval = np.linspace(0, 3, 200)
    y_eval = sol.sol(t_eval)

    # Plot this method's solution
    plt.plot(t_eval, y_eval[0], label=f'{method} Method')

# Add the analytical solution for comparison
if 'y_solution' in locals():
    plt.plot(t_points, y_points, 'k--', label='Analytical Solution')

plt.grid(True)
plt.xlabel('t')
plt.ylabel('y(t)')
plt.title('Numerical Solutions to dy/dt = 3t - 2y with y(0) = 4')
plt.legend()
plt.show()

# Compare computational efficiency
print("Performance comparison:")
for method in methods:
    sol = solve_ivp(differential_eq, t_span, y0, method=method)
    print(f"{method}: {sol.nfev} function evaluations, {sol.njev} Jacobian evaluations")

## Method of Separable Variables

### Example: First-Order Reaction Kinetics

**Analysis of First-Order Reaction Kinetics**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.optimize import curve_fit

# Symbolic verification that C0*exp(-k*t) satisfies dC/dt = -k*C
t, k, C0 = sp.symbols('t k C0', real=True, positive=True)
solution = C0*sp.exp(-k*t)
verification = sp.diff(solution, t) + k*solution
print(f"Verification (should equal zero): {sp.simplify(verification)}")

# Numerical implementation
def concentration(t, C0, k):
    """Concentration at time t for a first-order reaction."""
    return C0 * np.exp(-k * t)

def generate_data(t_values, C0=1.0, k=0.2, noise_level=0.03):
    """Synthetic measurements with multiplicative noise."""
    perfect_values = concentration(t_values, C0, k)
    noise = np.random.normal(0, noise_level, len(t_values))
    return perfect_values * (1 + noise)

# Parameter values
C0_true = 1.0                      # Initial concentration (mol/L)
k_values = [0.05, 0.1, 0.2, 0.4]   # Rate constants (min^-1)
t_max = 60                         # Maximum time (minutes)

fig, ax = plt.subplots(figsize=(12, 8))

# Theoretical decay curves
t_values = np.linspace(0, t_max, 200)
for k_rate in k_values:
    ax.plot(t_values, concentration(t_values, C0_true, k_rate),
            label=f'k = {k_rate} min$^{{-1}}$')

# Synthetic "experimental" data points
np.random.seed(42)  # For reproducibility
t_exp = np.linspace(0, t_max, 12)
C_exp = generate_data(t_exp, C0_true, 0.2)
ax.scatter(t_exp, C_exp, color='red', s=50,
           label='Experimental data (k=0.2)')

# Recover C0 and k by nonlinear least squares
params, _ = curve_fit(concentration, t_exp, C_exp, p0=(1.0, 0.1))
C0_fit, k_fit = params
ax.plot(t_values, concentration(t_values, C0_fit, k_fit), 'k--',
        label=f'Fitted curve: k={k_fit:.3f}, C0={C0_fit:.3f}')

ax.set_xlabel('Time (minutes)')
ax.set_ylabel('Concentration (mol/L)')
ax.legend(loc=(0.68, 0.28))  # keep clear of the inset
ax.grid(True, alpha=0.3)

# Inset with the linearized data: ln(C) = ln(C0) - k*t
ax_inset = fig.add_axes([0.55, 0.55, 0.3, 0.3])
ax_inset.scatter(t_exp, np.log(C_exp), color='red')
ax_inset.plot(t_values, np.log(C0_fit) - k_fit*t_values, 'k--')
ax_inset.set_title('Linearized Plot: ln(C) vs t')
ax_inset.grid(True, alpha=0.3)

plt.show()

### Example: Population Dynamics with Time-Dependent Harvesting

**Python implementation for ecological population model**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import sympy as sp

# Define symbolic variable for analytical solution verification
t_sym = sp.Symbol('t')

# Define the analytical solution
def analytical_solution(t):
    return 2 * sp.exp(t*sp.exp(t) - sp.exp(t) - t + 1)

# Define the differential equation
def population_model(t, P):
    return t * np.exp(t) * P - P

# Verify analytically by differentiating our solution
P_solution = analytical_solution(t_sym)
dP_dt = sp.diff(P_solution, t_sym)
verification = dP_dt + P_solution - t_sym * sp.exp(t_sym) * P_solution
print(f"Analytical verification (should be zero): {sp.simplify(verification)}")

# Numerical validation
t_span = (0, 2)
t_eval = np.linspace(0, 2, 100)
initial_condition = [2]

# Solve numerically using SciPy's ODE solver
numerical_sol = solve_ivp(population_model, t_span, initial_condition,
                          t_eval=t_eval, method='RK45',
                          rtol=1e-9, atol=1e-9)

# Calculate analytical solutions at the same points
analytical_vals = [float(analytical_solution(t).evalf()) for t in t_eval]

# Visualization
plt.figure(figsize=(12, 7))

# Plot both solutions
plt.plot(t_eval, analytical_vals, 'b-', label='Analytical Solution')
plt.plot(numerical_sol.t, numerical_sol.y[0], 'r--', label='Numerical Solution')

# Add reference lines
plt.axhline(y=2, color='g', linestyle=':', label='Initial Population P(0)=2')
plt.axvline(x=0, color='k', linestyle=':')

# Formatting
plt.grid(True, alpha=0.3)
plt.title('Population Dynamics with Controlled Harvesting')
plt.xlabel('Time (t)')
plt.ylabel('Population Size P(t)')
plt.legend()

# Add annotations for key mathematical insights
t_critical = 1.5
P_critical = float(analytical_solution(t_critical).evalf())
plt.annotate(f'Rapid growth\nphase', xy=(t_critical, P_critical),
             xytext=(t_critical-0.8, P_critical-15),
             arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

# Calculate error between analytical and numerical solutions
max_error = max(abs(np.array(analytical_vals) - numerical_sol.y[0]))
print(f"Maximum absolute error between solutions: {max_error:.2e}")

### Example: Newton's Law of Cooling

**SymPy implementation for Newton's Law of Cooling**

In [ ]:
import sympy as sp
import matplotlib.pyplot as plt
import numpy as np

# Define symbolic variables
t = sp.Symbol('t', real=True, positive=True)  # Time
T = sp.Function('T')(t)                       # Temperature as a function of time
T_a = sp.Symbol('T_a', real=True)             # Ambient temperature
T_0 = sp.Symbol('T_0', real=True)             # Initial temperature
k = sp.Symbol('k', real=True, positive=True)  # Cooling constant

# Define the differential equation from Newton's Law of Cooling
cooling_eq = sp.Eq(sp.diff(T, t), -k * (T - T_a))
print("Newton's Law of Cooling differential equation:")
print(sp.pretty(cooling_eq))

# Solve the differential equation
print("\nSolving the differential equation...")
T_general = sp.dsolve(cooling_eq, T)
print("\nGeneral solution:")
print(sp.pretty(T_general))

# For debugging: Print the structure of the general solution
print("\nStructure of the general solution:")
print(f"Type of T_general.rhs: {type(T_general.rhs)}")
print(f"T_general.rhs expression: {T_general.rhs}")

# Manually identify the constant in the expression
# From the printed expression, we can see the structure and extract the constant correctly
C1 = sp.Symbol('C1')  # We know SymPy uses C1 as the constant of integration

# Create an equation for the initial condition T(0) = T_0
# Let's manually substitute t=0 into the general solution
T_at_0 = T_general.rhs.subs(t, 0)
print("\nT(0) from general solution:")
print(sp.pretty(T_at_0))

# Create the equation for initial condition
initial_condition_eq = sp.Eq(T_at_0, T_0)
print("\nInitial condition equation:")
print(sp.pretty(initial_condition_eq))

# Manually solve for C1
# Looking at the general solution, we know it has the form: T_a + C1*exp(-k*t)
# At t=0, this becomes: T_a + C1 = T_0
# Therefore, C1 = T_0 - T_a
C1_value = T_0 - T_a
print("\nValue of integration constant C1:", sp.pretty(C1_value))

# Substitute the constant into the general solution
T_particular = T_general.subs(C1, C1_value)
print("\nParticular solution:")
print(sp.pretty(T_particular))

# Simplify the solution
T_solution = sp.simplify(T_particular.rhs)
print("\nFinal simplified solution:")
print(sp.pretty(sp.Eq(T, T_solution)))

# Verify the solution
print("\nVerifying the solution by substituting back into the original equation...")
original_eq_lhs = sp.diff(T_solution, t)
original_eq_rhs = -k * (T_solution - T_a)
verification = sp.simplify(original_eq_lhs - original_eq_rhs)
print("Verification result (should be zero):", verification)

# Calculate the half-life
# We want to solve for t_half where: T(t_half) = T_a + (T_0 - T_a)/2
t_half_sym = sp.Symbol('t_half', positive=True)
half_temp = T_a + (T_0 - T_a)/2
half_life_eq = sp.Eq(T_solution.subs(t, t_half_sym), half_temp)
print("\nHalf-life equation:")
print(sp.pretty(half_life_eq))

# Manually solve the half-life equation
# If T(t_half) = T_a + (T_0 - T_a)/2, then:
# T_a + (T_0 - T_a)*exp(-k*t_half) = T_a + (T_0 - T_a)/2
# (T_0 - T_a)*exp(-k*t_half) = (T_0 - T_a)/2
# exp(-k*t_half) = 1/2
# -k*t_half = ln(1/2) = -ln(2)
# t_half = ln(2)/k
t_half_simplified = sp.log(2)/k
print("\nThermal half-life:")
print(sp.pretty(sp.Eq(sp.Symbol('t_half'), t_half_simplified)))

# Demonstrate the use of the solution with specific values
# Let's use some realistic values for demonstration
k_val = 0.05      # Cooling constant (1/min)
T_0_val = 90      # Initial temperature
T_a_val = 22      # Ambient temperature

# Convert symbolic solution to a numerical function
T_numeric = sp.lambdify(
    (t, T_0, T_a, k),
    T_solution,
    "numpy"
)

# Calculate the half-life with these values
t_half_val = float(t_half_simplified.subs({k: k_val}))
print(f"\nWith k = {k_val} 1/min, the thermal half-life is {t_half_val:.2f} minutes")

# Create time points for plotting
time_points = np.linspace(0, 120, 200)  # 0 to 120 minutes

# Calculate temperature values
temp_values = T_numeric(time_points, T_0_val, T_a_val, k_val)

# Plot temperature versus time
plt.figure(figsize=(10, 6))
plt.plot(time_points, temp_values, 'b-', linewidth=2)

# Add reference lines
plt.axhline(y=T_a_val, color='r', linestyle='--', label=f'Ambient temperature ({T_a_val}C)')
half_temp_val = T_a_val + (T_0_val - T_a_val)/2
plt.axhline(y=half_temp_val, color='g', linestyle=':', label=f'Half-cooling temperature ({half_temp_val}C)')
plt.axvline(x=t_half_val, color='g', linestyle=':', label=f'Half-life ({t_half_val:.2f} min)')

# Add plot annotations
plt.annotate(
    f"T(t) = {T_a_val} + ({T_0_val} - {T_a_val})e^(-{k_val}t)",
    xy=(40, 80),
    xytext=(40, 80),
    bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.8)
)

# Demonstration of property: at t = t_half, T = T_a + (T_0 - T_a)/2
plt.scatter([t_half_val], [half_temp_val], color='red', s=100, zorder=5)
plt.plot([0, t_half_val], [half_temp_val, half_temp_val], 'g:', alpha=0.7)
plt.plot([t_half_val, t_half_val], [T_a_val, half_temp_val], 'g:', alpha=0.7)

# Format the plot
plt.xlabel('Time (minutes)', fontsize=14)
plt.ylabel('Temperature (C)', fontsize=14)
plt.title("Newton's Law of Cooling: Temperature vs. Time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

# Let's also print some key values from our solution
print(f"\nInitial temperature: {T_0_val}C")
print(f"Ambient temperature: {T_a_val}C")
print(f"After {t_half_val:.2f} minutes (one half-life), temperature: {half_temp_val:.2f}C")
print(f"After {2*t_half_val:.2f} minutes (two half-lives), temperature: {T_a_val + (T_0_val - T_a_val)/4:.2f}C")
print(f"After {3*t_half_val:.2f} minutes (three half-lives), temperature: {T_a_val + (T_0_val - T_a_val)/8:.2f}C")

# Calculate time to reach a specific temperature
target_temp = 45  # C, ideal drinking temperature

# Manually derive the time to reach the target temperature
# From our solution: T(t) = T_a + (T_0 - T_a)*exp(-k*t)
# If T(t) = target_temp, then:
# target_temp = T_a + (T_0 - T_a)*exp(-k*t)
# (target_temp - T_a)/(T_0 - T_a) = exp(-k*t)
# ln((target_temp - T_a)/(T_0 - T_a)) = -k*t
# t = -ln((target_temp - T_a)/(T_0 - T_a))/k

target_ratio = (target_temp - T_a_val)/(T_0_val - T_a_val)
time_to_target = -np.log(target_ratio)/k_val
print(f"\nTime to reach {target_temp}C: {time_to_target:.2f} minutes")

# Add target point to plot
plt.scatter([time_to_target], [target_temp], color='purple', s=100, zorder=5)
plt.annotate(
    f"Ideal drinking\ntemperature: {target_temp}C\nt = {time_to_target:.2f} min",
    xy=(time_to_target, target_temp),
    xytext=(time_to_target+10, target_temp+5),
    arrowprops=dict(arrowstyle="->", color="black")
)

plt.show()

# Example: Compare different cooling constants
# Define 3 different scenarios
scenarios = [
    {"name": "Ceramic mug", "k": 0.05, "color": "red"},
    {"name": "Insulated tumbler", "k": 0.012, "color": "green"},
    {"name": "Metal bottle", "k": 0.08, "color": "blue"}
]

# Create a new figure for comparison
plt.figure(figsize=(10, 6))

for scenario in scenarios:
    # Calculate temperature values using the symbolic solution
    temp_values = T_numeric(time_points, T_0_val, T_a_val, scenario["k"])

    # Calculate half-life
    t_half_scenario = np.log(2) / scenario["k"]

    # Plot temperature curve
    plt.plot(time_points, temp_values, color=scenario["color"],
             label=f"{scenario['name']} ($t1/2$ = {t_half_scenario:.2f} min)")

# Add ambient temperature line
plt.axhline(y=T_a_val, color='k', linestyle='--', label=f'Ambient temperature ({T_a_val}C)')

# Format the comparison plot
plt.xlabel('Time (minutes)', fontsize=14)
plt.ylabel('Temperature (C)', fontsize=14)
plt.title("Newton's Law of Cooling: Comparison of Different Cooling Constants")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Print the symbolic expressions of key equations for reference
print("\nKey symbolic equations:")
print("-----------------------")
print("Original differential equation:")
print(sp.pretty(cooling_eq))
print("\nFinal solution:")
print(sp.pretty(sp.Eq(T, T_solution)))
print("\nThermal half-life:")
print(sp.pretty(sp.Eq(sp.Symbol('t_half'), t_half_simplified)))

### Example: Radial Groundwater Flow to a Pumping Well

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import sympy as sp

# ====================================================================
# PART 1: Symbolic Derivation with SymPy
# ====================================================================
print("=" * 70)
print("PART 1: Symbolic Derivation of the Thiem Equation")
print("=" * 70)

r = sp.Symbol("r", positive=True)
h = sp.Function("h")
Q, K, b_aq = sp.symbols("Q K b", positive=True)
r_w, R = sp.symbols("r_w R", positive=True)
h_R = sp.Symbol("h_R")
h_0 = sp.Symbol("h_0")
T = sp.Symbol("T", positive=True)

# 1) Governing ODE: (1/r) d/dr [r dh/dr] = 0
ode = sp.Eq(sp.diff(r * sp.diff(h(r), r), r) / r, 0)
print("\n1) Governing ODE (radial Laplace in cylindrical coords):")
sp.pprint(ode)

# 2) General solution
general_sol = sp.dsolve(ode, h(r))
print("\n2) General solution:")
sp.pprint(general_sol)

# 3) First-order ODE from Darcy + continuity
# For a pumping well (Q > 0 extraction), flow is inward.
# The Darcy velocity is q_r = -K dh/dr (positive outward).
# Continuity: Q = 2*pi*r*b*(-q_r) = 2*pi*r*b*K*dh/dr
# => dh/dr = Q / (2*pi*r*b*K)
# Head increases with r (lowest at well), which is correct.
dh_dr_expr = Q / (2 * sp.pi * r * b_aq * K)
print("\n3) First-order ODE from Darcy + continuity (Q > 0 for pumping):")
sp.pprint(sp.Eq(sp.Derivative(h(r), r), dh_dr_expr))

# 4) Integrate: h(r) - h(r1) = Q/(2*pi*b*K) * ln(r/r1)
h_diff = sp.integrate(dh_dr_expr, (r, R, r))
thiem_expr = sp.simplify(h_R + h_diff)
print("\n4) Thiem equation h(r) = h(R) + Q/(2*pi*T)*ln(r/R):")
sp.pprint(sp.Eq(sp.Symbol("h(r)"), thiem_expr))

# Using h(R) = h_0 at the radius of influence:
# h(r) = h_0 + Q/(2*pi*T) * ln(r/R)
# Since r < R for points inside the cone, ln(r/R) < 0, so h(r) < h_0. Correct!

# 5) Drawdown: s(r) = h_0 - h(r) = -Q/(2*pi*T) * ln(r/R) = Q/(2*pi*T) * ln(R/r)
s_expr = (Q / (2 * sp.pi * b_aq * K)) * sp.ln(R / r)
print("\n5) Drawdown s(r) = h_0 - h(r) = Q/(2*pi*T)*ln(R/r):")
sp.pprint(sp.Eq(sp.Symbol("s(r)"), s_expr))

# Also in terms of well radius:
s_expr_rw = (Q / (2 * sp.pi * b_aq * K)) * sp.ln(R / r_w)
print("\n   Maximum drawdown at the well (r = r_w):")
sp.pprint(sp.Eq(sp.Symbol("s_w"), s_expr_rw))

# 6) Verify solution satisfies ODE
print("\n6) Verification:")
h_explicit = h_R + (Q / (2 * sp.pi * b_aq * K)) * sp.ln(r / R)
lhs_check = sp.simplify(sp.diff(r * sp.diff(h_explicit, r), r) / r)
if lhs_check == 0:
    print("   (1/r) d/dr[r dh/dr] = 0  ... ODE satisfied!")
else:
    print(f"   (1/r) d/dr[r dh/dr] = {lhs_check}  ... FAIL")

# 7) Verify flow rate conservation
q_r_sym = -K * sp.diff(h_explicit, r)
Q_total = sp.simplify(-2 * sp.pi * r * b_aq * q_r_sym)
print(f"   Flow rate through cylinder = {Q_total}")
print(f"   Equals pumping rate Q: {sp.simplify(Q_total - Q) == 0}")

# ====================================================================
# PART 2: Numerical Evaluation
# ====================================================================
print("\n" + "=" * 70)
print("PART 2: Numerical Evaluation and Visualization")
print("=" * 70)

Q_val = 0.025
K_val = 1.5e-4
b_val = 10.0
r_w_val = 0.15
R_val = 500.0
h_0_val = 50.0
T_val = K_val * b_val

print(f"\nAquifer Parameters:")
print(f"  Q  = {Q_val*1000:.0f} L/s ({Q_val} m3/s)")
print(f"  K  = {K_val:.1e} m/s")
print(f"  b  = {b_val:.0f} m")
print(f"  T  = K*b = {T_val:.1e} m2/s")
print(f"  rw = {r_w_val} m")
print(f"  R  = {R_val:.0f} m (radius of influence)")
print(f"  h0 = {h_0_val} m (undisturbed head)")

# Lambdified functions (correct sign convention)
# h(r) = h_0 + Q/(2*pi*b*K) * ln(r/R)  [head, lowest at well]
# s(r) = Q/(2*pi*b*K) * ln(R/r)          [drawdown, highest at well]
h_num = sp.lambdify(
    (r, Q, b_aq, K, R, h_R),
    h_R + (Q / (2 * sp.pi * b_aq * K)) * sp.ln(r / R),
    modules="numpy"
)
s_num = sp.lambdify(
    (r, Q, b_aq, K, R),
    (Q / (2 * sp.pi * b_aq * K)) * sp.ln(R / r),
    modules="numpy"
)

r_lin = np.linspace(r_w_val, R_val, 1000)
r_log = np.logspace(np.log10(r_w_val), np.log10(R_val), 1000)
h_arr = h_num(r_lin, Q_val, b_val, K_val, R_val, h_0_val)
s_arr = s_num(r_lin, Q_val, b_val, K_val, R_val)

print("\nDrawdown at selected distances:")
print(f"  {'r [m]':>10}  {'s [m]':>10}  {'h [m]':>10}")
print("  " + "-" * 34)
for rc in [r_w_val, 1, 5, 10, 50, 100, 250, 500]:
    sv = s_num(rc, Q_val, b_val, K_val, R_val)
    hv = h_num(rc, Q_val, b_val, K_val, R_val, h_0_val)
    print(f"  {rc:>10.2f}  {sv:>10.2f}  {hv:>10.2f}")

# ====================================================================
# FIGURE 1: 4-panel visualization
# ====================================================================
fig = plt.figure(figsize=(16, 14))
fig.suptitle(
    "Thiem Equation: Steady-State Radial Flow to a Pumping Well\n"
    r"($Q$=25 L/s, $K$=1.5$\times 10^{-4}$ m/s, $b$=10 m, "
    r"$T$=1.5$\times 10^{-3}$ m$^2$/s)",
    fontsize=14, fontweight="bold", y=0.98
)

# Panel (a): Head vs distance
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(r_lin, h_arr, "b-", linewidth=2, label=r"$h(r)$")
ax1.axhline(h_0_val, color="gray", ls="--", alpha=0.7,
            label=r"$h_0 = $" + f"{h_0_val} m (undisturbed)")
ax1.fill_between(r_lin, h_arr, h_0_val, alpha=0.15, color="steelblue",
                 label="Drawdown zone")
ax1.set_xlabel("Distance from well, $r$ [m]", fontsize=11)
ax1.set_ylabel("Hydraulic head, $h$ [m]", fontsize=11)
ax1.set_title("(a) Hydraulic Head vs. Distance", fontweight="bold")
ax1.legend(loc="lower right", fontsize=9)
ax1.set_xlim([0, R_val])
ax1.grid(True, alpha=0.3)

# Panel (b): Drawdown semi-log
ax2 = fig.add_subplot(2, 2, 2)
s_log_arr = s_num(r_log, Q_val, b_val, K_val, R_val)
ax2.semilogx(r_log, s_log_arr, "r-", linewidth=2,
             label=r"$s(r) = \frac{Q}{2\pi T}\ln\!\left(\frac{R}{r}\right)$")
for rm in [r_w_val, 1, 10, 100]:
    sm = s_num(rm, Q_val, b_val, K_val, R_val)
    ax2.plot(rm, sm, "ko", markersize=6)
    ax2.annotate(
        f"r={rm} m\ns={sm:.1f} m", xy=(rm, sm), fontsize=8,
        xytext=(15, -15), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8)
    )
ax2.set_xlabel("Distance, $r$ [m] (log scale)", fontsize=11)
ax2.set_ylabel("Drawdown, $s$ [m]", fontsize=11)
ax2.set_title("(b) Drawdown vs. Distance (semi-log)", fontweight="bold")
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(True, alpha=0.3, which="both")

# Panel (c): Cross-section
ax3 = fig.add_subplot(2, 2, 3)
ax3.axhline(0, color="saddlebrown", linewidth=3, label="Confining layers")
ax3.axhline(-b_val, color="saddlebrown", linewidth=3)
ax3.fill_between([-R_val, R_val], 0, 2, color="burlywood", alpha=0.4)
ax3.fill_between([-R_val, R_val], -b_val, -b_val - 2, color="burlywood", alpha=0.4)
rp = np.linspace(r_w_val, R_val, 500)
hp = h_num(rp, Q_val, b_val, K_val, R_val, h_0_val)
ep = hp - h_0_val  # negative values (drawdown below initial level)
ax3.plot(rp, ep, "b-", linewidth=2)
ax3.plot(-rp, ep, "b-", linewidth=2, label="Piezometric surface")
ax3.fill_between(rp, 0, ep, alpha=0.2, color="steelblue")
ax3.fill_between(-rp, 0, ep, alpha=0.2, color="steelblue")
ww = R_val * 0.012
ax3.fill_between([-ww, ww], -b_val, 2, color="gray", alpha=0.6, zorder=5)
ax3.annotate("Well", xy=(0, 1), fontsize=9, ha="center", fontweight="bold")
# Arrows showing flow toward well
for ra in [50, 150, 300]:
    for sgn in [-1, 1]:
        ax3.annotate("", xy=(sgn * (ra - 30), -b_val / 2),
                     xytext=(sgn * (ra + 30), -b_val / 2),
                     arrowprops=dict(arrowstyle="->", color="navy", lw=1.5))
ax3.set_xlabel("Distance from well [m]", fontsize=11)
ax3.set_ylabel("Elevation rel. to initial head [m]", fontsize=11)
ax3.set_title("(c) Cross-Section: Cone of Depression", fontweight="bold")
ax3.set_xlim([-R_val, R_val])
s_max = s_num(r_w_val, Q_val, b_val, K_val, R_val)
ax3.set_ylim([-(s_max + 2), 3])
ax3.legend(loc="upper right", fontsize=8)
ax3.grid(True, alpha=0.3)

# Panel (d): Plan view
ax4 = fig.add_subplot(2, 2, 4)
xg = np.linspace(-R_val, R_val, 500)
X, Y = np.meshgrid(xg, xg)
Rg = np.sqrt(X**2 + Y**2)
Rg_c = np.clip(Rg, r_w_val, R_val)
Sg = s_num(Rg_c, Q_val, b_val, K_val, R_val)
Sg[Rg < r_w_val] = np.nan
Sg[Rg > R_val] = np.nan
levels = np.arange(0, np.nanmax(Sg) + 0.5, 1.0)
cf = ax4.contourf(X, Y, Sg, levels=levels, cmap="Blues")
cl = ax4.contour(X, Y, Sg, levels=levels[::2], colors="navy", linewidths=0.5, alpha=0.6)
ax4.clabel(cl, inline=True, fontsize=7, fmt="%.0f m")
ax4.add_patch(Circle((0, 0), r_w_val * 30, color="red", zorder=5))
ax4.annotate("Well", xy=(0, 0), xytext=(20, 20), textcoords="offset points",
             fontsize=9, fontweight="bold", color="red",
             arrowprops=dict(arrowstyle="->", color="red"))
plt.colorbar(cf, ax=ax4, label="Drawdown, $s$ [m]", shrink=0.9)
ax4.set_xlabel("$x$ [m]", fontsize=11)
ax4.set_ylabel("$y$ [m]", fontsize=11)
ax4.set_title("(d) Plan View: Drawdown Contours", fontweight="bold")
ax4.set_aspect("equal")
ax4.grid(True, alpha=0.2)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("thiem_equation_1.png", dpi=200, bbox_inches="tight")

# ====================================================================
# FIGURE 2: Sensitivity Analysis
# ====================================================================
fig2, (ax_s1, ax_s2) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle("Sensitivity Analysis: Pumping Rate and Transmissivity",
              fontsize=13, fontweight="bold")

Q_list = [0.010, 0.015, 0.020, 0.025, 0.030]
colors_q = plt.cm.Reds(np.linspace(0.3, 0.9, len(Q_list)))
for qv, col in zip(Q_list, colors_q):
    ax_s1.semilogx(r_log, s_num(r_log, qv, b_val, K_val, R_val),
                   color=col, linewidth=2, label=f"Q = {qv*1000:.0f} L/s")
ax_s1.set_xlabel("$r$ [m]", fontsize=11)
ax_s1.set_ylabel("$s$ [m]", fontsize=11)
ax_s1.set_title("(a) Varying Pumping Rate $Q$", fontweight="bold")
ax_s1.legend(fontsize=9)
ax_s1.grid(True, alpha=0.3, which="both")

K_list = [0.5e-4, 1.0e-4, 1.5e-4, 2.0e-4, 3.0e-4]
colors_k = plt.cm.Blues(np.linspace(0.3, 0.9, len(K_list)))
for kv, col in zip(K_list, colors_k):
    Tv = kv * b_val
    ax_s2.semilogx(r_log, s_num(r_log, Q_val, b_val, kv, R_val),
                   color=col, linewidth=2, label=f"T = {Tv:.1e} m$^2$/s")
ax_s2.set_xlabel("$r$ [m]", fontsize=11)
ax_s2.set_ylabel("$s$ [m]", fontsize=11)
ax_s2.set_title("(b) Varying Transmissivity $T$", fontweight="bold")
ax_s2.legend(fontsize=9)
ax_s2.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## Homogeneous Differential Equations

### Visualization of Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

def implicit_function(y, x, C):
    """The implicit function defined by our solution"""
    return y * np.exp(y/x) - C/x**3

def find_y_for_x(x_val, C_val):
    """Find a y value satisfying the implicit equation for given x, C."""
    if x_val == 0:
        return np.nan

    # Initial guess for y
    y_guess = x_val

    # Use fsolve and keep the root only if the iteration converged
    y_sol, _, ier, _ = fsolve(lambda y: implicit_function(y, x_val, C_val),
                              y_guess, full_output=True)
    return y_sol[0] if ier == 1 else np.nan

# Create a range of x values for plotting
x_values = np.linspace(-3, 3, 300)
x_values = x_values[x_values != 0]  # Remove x=0 since it's a singularity

# Constants for different solution curves
C_values = [0.5, 1, 2, 5, 10, -1, -2]
colors = plt.cm.viridis(np.linspace(0, 1, len(C_values)))

plt.figure(figsize=(10, 8))

# Plot solution curves for each constant C
for i, C in enumerate(C_values):
    y_values = [find_y_for_x(x, C) for x in x_values]
    plt.plot(x_values, y_values, label=f'C = {C}', color=colors[i], linewidth=2)

plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.grid(True, linestyle='--', alpha=0.7)
plt.title('Solution Curves for the Homogeneous Equation')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.ylim(-5, 5)
plt.tight_layout()
plt.show()

## Exact Differential Equations

### Visualization of Exact Equation Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def solution_function(x, y, c):
    """The solution function F(x,y) = c for our exact equation"""
    return x*y**2 - x**2*y - c

# Create a grid of x and y values
x_range = np.linspace(-4, 4, 400)
y_range = np.linspace(-4, 4, 400)
X, Y = np.meshgrid(x_range, y_range)

# Values of the constant C for different solution curves
c_values = [-5, -2, 0, 2, 5, 10, 15]

plt.figure(figsize=(10, 8))

# Create a contour plot for each value of C
for c in c_values:
    Z = solution_function(X, Y, c)
    contour = plt.contour(X, Y, Z, levels=[0],
                         colors=[plt.cm.plasma((c+5)/20)],
                         linewidths=2)

plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.title('Solution Curves for the Exact Differential Equation')
plt.xlabel('x')
plt.ylabel('y')

# Create a custom colorbar
norm = Normalize(vmin=min(c_values), vmax=max(c_values))
sm = ScalarMappable(cmap='plasma', norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca()) # Pass the current axes to colorbar
cbar.set_label('Constant C')

# Add text annotations for each curve
for c in c_values:
    Z = solution_function(X, Y, c)
    contour = plt.contour(X, Y, Z, levels=[0], alpha=0)
    plt.clabel(contour, inline=True, fmt=f'C = {c}', fontsize=9)

plt.tight_layout()
plt.show()

## First-Order Linear Differential Equations

### Example: Temperature Variation in a Thermal System

**Symbolic and numerical solution of $dT/dx = T/(x\ln x)$ with $T(2) = 5$, illustrating the integrating-factor method on a rod of variable cross-section**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp

# Symbolic verification
x_sym = sp.Symbol('x', real=True, positive=True)
T_sym = sp.Function('T')(x_sym)

# Define the differential equation: dT/dx = T/(x*ln(x))
diff_eq = sp.Eq(sp.diff(T_sym, x_sym), T_sym / (x_sym * sp.ln(x_sym)))
print("Differential equation:")
print(sp.pretty(diff_eq))

# Solve analytically
T_general = sp.dsolve(diff_eq, T_sym)
print("\nGeneral solution:")
print(sp.pretty(T_general))

# The general solution is T = C * ln(x)
# Apply initial condition T(2) = 5
C_sym = sp.Symbol('C')
T_solution = C_sym * sp.ln(x_sym)
T_at_2 = T_solution.subs(x_sym, 2)
C_value = sp.solve(sp.Eq(T_at_2, 5), C_sym)[0]
print(f"\nFrom T(2) = 5: C = {C_value} = {float(C_value):.4f}")

# Particular solution
T_particular = C_value * sp.ln(x_sym)
print("\nParticular solution:")
print(f"T(x) = {C_value} * ln(x) = {float(C_value):.4f} * ln(x)")

# Verify the solution
verification = sp.diff(T_particular, x_sym) - T_particular / (x_sym * sp.ln(x_sym))
print(f"\nVerification (should be zero): {sp.simplify(verification)}")

# Convert to numerical function
T_numerical = sp.lambdify(x_sym, T_particular, 'numpy')

# Create visualization
x_values = np.linspace(1.01, 10, 200)  # Start > 1 to avoid ln(1)=0
T_values = T_numerical(x_values)

plt.figure(figsize=(12, 5))

# Plot 1: Temperature vs Distance
plt.subplot(1, 2, 1)
plt.plot(x_values, T_values, 'b-', linewidth=2, label='T(x)')
plt.scatter([2], [5], color='red', s=100, zorder=5, label='Initial condition T(2)=5')
plt.xlabel('Position x (distance from end)')
plt.ylabel('Temperature T(x)')
plt.title('Heat Distribution in Variable Cross-Section Rod')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 2: Cross-sectional area variation A(x) = k/ln(x)
plt.subplot(1, 2, 2)
k_val = 1  # Normalized constant
A_values = k_val / np.log(x_values)
plt.plot(x_values, A_values, 'g-', linewidth=2)
plt.xlabel('Position x')
plt.ylabel('Cross-sectional Area A(x)')
plt.title('Variable Cross-Section: A(x) = k/ln(x)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Numerical verification using scipy
def heat_ode(x, T):
    """ODE in the form dT/dx = f(x, T)"""
    if x <= 1:
        return 0
    return T / (x * np.log(x))

# Solve numerically, starting from the known condition T(2) = 5
x_span = (2, 10)
x_eval = np.linspace(2, 10, 200)
sol_numeric = solve_ivp(heat_ode, x_span, [5], t_eval=x_eval,
                        method='RK45', rtol=1e-8, atol=1e-10)

# Compare analytical vs numerical
T_analytical = T_numerical(sol_numeric.t)
error = np.abs(T_analytical - sol_numeric.y[0])

print(f"\nMaximum error between analytical and numerical: {np.max(error):.2e}")
print(f"Temperature at x=2: {T_numerical(2):.4f} (should be 5.0000)")
print(f"Temperature at x=10: {T_numerical(10):.4f}")

### Example: Epidemic Modeling with the SIR System

#### Python Implementation of SIR Model

**SIR epidemic model: numerical integration of \cref{eq:ch2-sir-S**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

def sir_model(y, t, beta, gamma):
    """
    SIR model: dy/dt = [dS/dt, dI/dt, dR/dt]
    y = [S, I, R]
    """
    S, I, R = y
    N = S + I + R  # Total population

    dS_dt = -beta * S * I / N
    dI_dt = beta * S * I / N - gamma * I
    dR_dt = gamma * I

    return [dS_dt, dI_dt, dR_dt]

# Parameters
N = 10000  # Total population
I0 = 10    # Initial infected
R0_init = 0
S0 = N - I0 - R0_init
y0 = [S0, I0, R0_init]

# Disease parameters
beta = 0.4   # Transmission rate
gamma = 0.1  # Recovery rate
R0 = beta / gamma  # Basic reproduction number

print(f"Basic reproduction number R0 = {R0:.2f}")
# Classical SIR peak formula (S0 ~ N): I_max/N = 1 - (1 + ln R0)/R0
I_peak_pred = N * (1 - (1 + np.log(R0)) / R0)
print(f"Predicted peak infections: {I_peak_pred:.0f}")

# Time points
t = np.linspace(0, 160, 1000)

# Solve ODE system
solution = odeint(sir_model, y0, t, args=(beta, gamma))
S = solution[:, 0]
I = solution[:, 1]
R = solution[:, 2]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: SIR dynamics
ax = axes[0]
ax.plot(t, S, 'b-', label='Susceptible', linewidth=2)
ax.plot(t, I, 'r-', label='Infected', linewidth=2)
ax.plot(t, R, 'g-', label='Recovered', linewidth=2)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Number of individuals')
ax.set_title(f'SIR Model ($R_0 = {R0:.2f}$)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Compare different reproduction numbers
ax = axes[1]
R0_values = [0.8, 1.5, 2.5]
colors = ['green', 'orange', 'red']

for R0_test, color in zip(R0_values, colors):
    beta_test = R0_test * gamma
    sol = odeint(sir_model, y0, t, args=(beta_test, gamma))
    ax.plot(t, sol[:, 1], label=f'$R_0 = {R0_test}$', color=color, linewidth=2)

ax.set_xlabel('Time (days)')
ax.set_ylabel('Number of infected')
ax.set_title('Effect of Basic Reproduction Number on Peak Infection')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
peak_infected_idx = np.argmax(I)
peak_infected = I[peak_infected_idx]
peak_time = t[peak_infected_idx]

print(f"\nEpidemic Statistics:")
print(f"Peak infections: {peak_infected:.0f} at day {peak_time:.1f}")
print(f"Final susceptible: {S[-1]:.0f}")
print(f"Final recovered/immune: {R[-1]:.0f}")
print(f"Attack rate: {100 * (1 - S[-1]/N):.1f}%")

## Graphical Approaches to Differential Equations

### Direction Fields

#### Python Implementation of Direction Fields

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def f(t, y):
    """Example: dy/dt = y(1-y)"""
    return y * (1 - y)

# Create grid of points
t_vals = np.linspace(0, 5, 20)
y_vals = np.linspace(-0.5, 1.5, 20)
t_grid, y_grid = np.meshgrid(t_vals, y_vals)

# Calculate slopes at each point
slopes = f(t_grid, y_grid)
angles = np.arctan(slopes)

# Create figure
plt.figure(figsize=(10, 6))

# Plot direction field (quiver plot)
plt.quiver(t_grid, y_grid, np.cos(angles), np.sin(angles),
           slopes, cmap='RdYlGn', scale=30, width=0.003)

# Plot several solution curves with different initial conditions
initial_conditions = [-0.3, 0.2, 0.5, 0.8, 1.2]
for y0 in initial_conditions:
    sol = solve_ivp(f, [0, 5], [y0], dense_output=True, method='RK45')
    t_dense = np.linspace(0, 5, 200)
    y_dense = sol.sol(t_dense)[0]
    plt.plot(t_dense, y_dense, 'k-', alpha=0.7, linewidth=1.5)

# Equilibrium lines
plt.axhline(y=0, color='blue', linestyle='--', linewidth=2, label='Equilibrium: y=0')
plt.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Equilibrium: y=1')

plt.xlabel('t')
plt.ylabel('y')
plt.title('Direction Field for dy/dt = y(1-y)')
plt.xlim(0, 5)
plt.ylim(-0.5, 1.5)
plt.colorbar(label='Slope dy/dt')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Equilibrium Points and Stability Analysis

#### Phase Line Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def f(y):
    """Autonomous system: dy/dt = y^2 - 2y"""
    return y**2 - 2*y

def f_prime(y):
    """Derivative for stability analysis"""
    return 2*y - 2

# Find equilibrium points by solving f(y) = 0
equilibria = [0, 2]  # From y(y-2) = 0

# Determine stability
print("Equilibrium Analysis:")
for y_eq in equilibria:
    deriv = f_prime(y_eq)
    if deriv < 0:
        stability = "STABLE (sink)"
    elif deriv > 0:
        stability = "UNSTABLE (source)"
    else:
        stability = "SEMISTABLE"
    print(f"y* = {y_eq}: f'({y_eq}) = {deriv:.1f} --> {stability}")

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: f(y) vs y
ax = axes[0, 0]
y_range = np.linspace(-1, 3, 300)
f_vals = f(y_range)
ax.plot(y_range, f_vals, 'b-', linewidth=2, label='f(y) = y^2 - 2y')
ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='-', alpha=0.3)
for y_eq in equilibria:
    ax.plot(y_eq, 0, 'ro', markersize=10)
ax.set_xlabel('y')
ax.set_ylabel('f(y) = dy/dt')
ax.set_title('Autonomous ODE: dy/dt = y^2 - 2y')
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Phase line (vertical; arrows follow the sign of f)
ax = axes[0, 1]
ax.axvline(x=0.5, color='k', linewidth=2)
for y_arrow in [-0.8, 0.6, 1.1, 1.6, 2.4]:
    step = 0.25 if f(y_arrow) > 0 else -0.25
    ax.annotate('', xy=(0.5, y_arrow + step), xytext=(0.5, y_arrow),
                arrowprops=dict(arrowstyle='-|>', color='purple', lw=2))
for y_eq in equilibria:
    stable = f_prime(y_eq) < 0
    ax.plot(0.5, y_eq, 'o' if stable else 'x',
            color='green' if stable else 'red',
            markersize=14, markeredgewidth=3)

ax.set_xlim(0, 1)
ax.set_ylim(-1, 3)
ax.set_yticks(equilibria)
ax.set_yticklabels([f'y* = {y}' for y in equilibria])
ax.set_xticks([])
ax.set_title('Phase Line (o = stable, x = unstable)')

# Plot 3: Direction field for (t, y)
ax = axes[1, 0]
t_vals = np.linspace(0, 3, 15)
y_vals = np.linspace(-0.5, 2.5, 15)
t_grid, y_grid = np.meshgrid(t_vals, y_vals)
slopes = f(y_grid)

# Normalize for visualization
max_slope = np.max(np.abs(slopes))
slopes_norm = slopes / (max_slope + 0.1)
ax.quiver(t_grid, y_grid, np.ones_like(slopes_norm), slopes_norm,
          color='purple', scale=30, width=0.005)

# Plot equilibrium lines
for y_eq in equilibria:
    ax.axhline(y=y_eq, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax.set_xlabel('t')
ax.set_ylabel('y')
ax.set_title('Slope Field in (t, y) plane')
ax.grid(True, alpha=0.3)

# Plot 4: Sample trajectories
ax = axes[1, 1]
y0_vals = [-0.5, 0.5, 1.5, 2.5]
t_span = (0, 3)
t_eval = np.linspace(0, 3, 200)

for y0 in y0_vals:
    sol = solve_ivp(lambda t, y: f(y), t_span, [y0], t_eval=t_eval, method='RK45')
    ax.plot(sol.t, sol.y[0], linewidth=2, label=f'y(0) = {y0}')

for y_eq in equilibria:
    ax.axhline(y=y_eq, color='red', linestyle='--', alpha=0.5)

ax.set_xlabel('t')
ax.set_ylabel('y(t)')
ax.set_title('Solution Curves')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Numerical Approaches to Differential Equations

### Classical Fourth-Order Runge–Kutta (RK4)

#### Python Implementation of Numerical Methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(t, y):
    """Test ODE: dy/dt = -2*t*y, y(0) = 1"""
    return -2 * t * y

def y_analytical(t):
    """Analytical solution: y = exp(-t^2)"""
    return np.exp(-t**2)

def forward_euler(f, t_span, y0, h):
    """Forward Euler method"""
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros_like(t)
    y[0] = y0

    for n in range(len(t) - 1):
        y[n+1] = y[n] + h * f(t[n], y[n])

    return t, y

def improved_euler(f, t_span, y0, h):
    """Improved Euler (Heun) method"""
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros_like(t)
    y[0] = y0

    for n in range(len(t) - 1):
        y_pred = y[n] + h * f(t[n], y[n])
        y[n+1] = y[n] + 0.5 * h * (f(t[n], y[n]) + f(t[n+1], y_pred))

    return t, y

def rk4(f, t_span, y0, h):
    """Classical 4th-order Runge-Kutta method"""
    t0, tf = t_span
    t = np.arange(t0, tf + h, h)
    y = np.zeros_like(t)
    y[0] = y0

    for n in range(len(t) - 1):
        k1 = h * f(t[n], y[n])
        k2 = h * f(t[n] + 0.5*h, y[n] + 0.5*k1)
        k3 = h * f(t[n] + 0.5*h, y[n] + 0.5*k2)
        k4 = h * f(t[n] + h, y[n] + k3)
        y[n+1] = y[n] + (k1 + 2*k2 + 2*k3 + k4) / 6

    return t, y

# Initial conditions
t_span = (0, 2)
y0 = 1

# Solve with different step sizes
h_values = [0.2, 0.1, 0.05]
t_analytical = np.linspace(0, 2, 500)
y_analytical_vals = y_analytical(t_analytical)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Forward Euler
ax = axes[0, 0]
ax.plot(t_analytical, y_analytical_vals, 'k-', linewidth=2.5, label='Analytical')
for h in h_values:
    t, y = forward_euler(f, t_span, y0, h)
    ax.plot(t, y, 'o-', markersize=4, label=f'h = {h}')
ax.set_title('Forward Euler Method (1st Order)')
ax.set_xlabel('t')
ax.set_ylabel('y(t)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Improved Euler
ax = axes[0, 1]
ax.plot(t_analytical, y_analytical_vals, 'k-', linewidth=2.5, label='Analytical')
for h in h_values:
    t, y = improved_euler(f, t_span, y0, h)
    ax.plot(t, y, 's-', markersize=4, label=f'h = {h}')
ax.set_title('Improved Euler Method (2nd Order)')
ax.set_xlabel('t')
ax.set_ylabel('y(t)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: RK4
ax = axes[1, 0]
ax.plot(t_analytical, y_analytical_vals, 'k-', linewidth=2.5, label='Analytical')
for h in h_values:
    t, y = rk4(f, t_span, y0, h)
    ax.plot(t, y, '^-', markersize=4, label=f'h = {h}')
ax.set_title('RK4 Method (4th Order)')
ax.set_xlabel('t')
ax.set_ylabel('y(t)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Convergence Analysis
ax = axes[1, 1]
h_test = np.array([0.1, 0.05, 0.025, 0.0125])
errors_euler = []
errors_improved = []
errors_rk4 = []

for h in h_test:
    t_e, y_e = forward_euler(f, t_span, y0, h)
    t_i, y_i = improved_euler(f, t_span, y0, h)
    t_r, y_r = rk4(f, t_span, y0, h)

    y_analytical_e = y_analytical(t_e)
    y_analytical_i = y_analytical(t_i)
    y_analytical_r = y_analytical(t_r)

    errors_euler.append(np.max(np.abs(y_e - y_analytical_e)))
    errors_improved.append(np.max(np.abs(y_i - y_analytical_i)))
    errors_rk4.append(np.max(np.abs(y_r - y_analytical_r)))

ax.loglog(h_test, errors_euler, 'o-', linewidth=2, markersize=8, label='Euler (O(h))')
ax.loglog(h_test, errors_improved, 's-', linewidth=2, markersize=8, label='Improved Euler (O(h^2))')
ax.loglog(h_test, errors_rk4, '^-', linewidth=2, markersize=8, label='RK4 (O(h^4))')

# Reference lines for convergence rates
ax.loglog(h_test, errors_euler[0] * (h_test / h_test[0]), 'o--', color='gray', alpha=0.5)
ax.loglog(h_test, errors_improved[0] * (h_test / h_test[0])**2, 's--', color='gray', alpha=0.5)
ax.loglog(h_test, errors_rk4[0] * (h_test / h_test[0])**4, '^--', color='gray', alpha=0.5)

ax.set_xlabel('Step size h')
ax.set_ylabel('Maximum absolute error')
ax.set_title('Convergence Analysis')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Print convergence rates
print("Convergence Rates (slope in log-log plot):")
for method, errors in [('Euler', errors_euler),
                       ('Improved Euler', errors_improved),
                       ('RK4', errors_rk4)]:
    log_h = np.log(h_test)
    log_err = np.log(np.array(errors))
    rate = np.polyfit(log_h, log_err, 1)[0]
    print(f"{method}: {rate:.2f} (expected 1, 2, 4 respectively)")

## Stability Analysis of Equilibrium Points

### Stability Analysis Example: Logistic Equation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def logistic(y, r, K):
    """Logistic equation: dy/dt = r*y*(1 - y/K)"""
    return r * y * (1 - y / K)

def logistic_derivative(y, r, K):
    """Derivative of logistic function"""
    return r * (1 - 2*y / K)

# Parameters
r = 0.5
K = 100

# Plot phase portrait
y = np.linspace(-10, 150, 1000)
f_y = logistic(y, r, K)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Phase line plot
ax1.plot(y, f_y, 'b-', linewidth=2)
ax1.axhline(0, color='k', linestyle='-', alpha=0.3)
ax1.plot([0, K], [0, 0], 'ro', markersize=8)
ax1.fill_between(y, 0, f_y, where=(f_y>0), alpha=0.3, color='green', label='dy/dt > 0')
ax1.fill_between(y, 0, f_y, where=(f_y<0), alpha=0.3, color='red', label='dy/dt < 0')
ax1.arrow(25, 0, 0, 5, head_width=2, head_length=1, fc='green', ec='green')
ax1.arrow(125, 0, 0, -5, head_width=2, head_length=1, fc='red', ec='red')
ax1.set_xlabel('y')
ax1.set_ylabel('dy/dt')
ax1.set_title('Phase Line Diagram')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Stability analysis
y_eq = np.array([0, K])
derivatives = logistic_derivative(y_eq, r, K)
colors = ['red' if d > 0 else 'green' for d in derivatives]
ax2.bar(range(len(y_eq)), derivatives, color=colors, alpha=0.7)
ax2.axhline(0, color='k', linestyle='-', linewidth=1)
ax2.set_xticks(range(len(y_eq)))
ax2.set_xticklabels([f'y* = {y}' for y in y_eq])
ax2.set_ylabel("f'(y*)")
ax2.set_title('Stability at Equilibria')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print stability analysis
print("Equilibrium Analysis for Logistic Equation")
print(f"Parameters: r = {r}, K = {K}")
print(f"\nEquilibrium at y = 0:")
print(f"  f'(0) = {logistic_derivative(0, r, K):.4f} > 0 -> UNSTABLE")
print(f"\nEquilibrium at y = K:")
print(f"  f'(K) = {logistic_derivative(K, r, K):.4f} < 0 -> STABLE")

## Bifurcation Analysis with Python

### Example: Transcritical Bifurcation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(y, mu):
    """Transcritical bifurcation: dy/dt = mu*y - y^2"""
    return mu * y - y**2

def f_prime(y, mu):
    """Derivative: f'(y) = mu - 2*y"""
    return mu - 2*y

# Combined figure: four phase portraits + bifurcation diagram
fig = plt.figure(figsize=(12, 13))
gs = fig.add_gridspec(3, 2)

# Panels (a)-(d): phase portraits for representative mu values
for idx, mu in enumerate([-1, -0.5, 0.5, 1.5]):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    y = np.linspace(-2, 3, 200)
    f_vals = f(y, mu)

    ax.plot(y, f_vals, 'b-', linewidth=2)
    ax.axhline(0, color='k', linestyle='-', alpha=0.3)
    ax.axvline(0, color='k', linestyle='-', alpha=0.3)

    # Mark equilibria: green circle = stable, red cross = unstable
    eq_points = [0, mu] if mu != 0 else [0]
    for eq in eq_points:
        stab = f_prime(eq, mu)
        color = 'green' if stab < 0 else 'red'
        marker = 'o' if stab < 0 else 'x'
        ax.plot(eq, 0, marker, markersize=10, color=color)

    ax.set_xlabel('y')
    ax.set_ylabel('dy/dt')
    ax.set_title(f'Phase Portrait: mu = {mu}')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1, 1)

# Panel (e): bifurcation diagram across the bottom row
mu_range = np.linspace(-2, 3, 500)
stable_branches = []
unstable_branches = []

for mu in mu_range:
    # Branch y* = 0: stable for mu < 0, unstable for mu > 0
    (stable_branches if mu < 0 else unstable_branches).append((mu, 0))
    # Branch y* = mu: stable for mu > 0, unstable for mu < 0
    (stable_branches if mu > 0 else unstable_branches).append((mu, mu))

ax = fig.add_subplot(gs[2, :])
stable_mu, stable_y = zip(*stable_branches)
ax.plot(stable_mu, stable_y, 'g-', linewidth=2.5, label='Stable')
unstable_mu, unstable_y = zip(*unstable_branches)
ax.plot(unstable_mu, unstable_y, 'r--', linewidth=2.5, label='Unstable')

ax.axvline(0, color='k', linestyle=':', alpha=0.5, linewidth=1)
ax.plot(0, 0, 'ko', markersize=8)
ax.set_xlabel('Parameter mu', fontsize=12)
ax.set_ylabel('Equilibrium y*', fontsize=12)
ax.set_title('Transcritical Bifurcation Diagram', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_xlim(-2, 3)
ax.set_ylim(-2, 3)

plt.tight_layout()
plt.show()

print("Bifurcation Analysis: dy/dt = mu*y - y^2")
print("Bifurcation point: mu = 0")
print("\nFor mu < 0:")
print("  - y = 0 is STABLE (attractor)")
print("  - y = mu < 0 is UNSTABLE")
print("\nFor mu > 0:")
print("  - y = 0 is UNSTABLE")
print("  - y = mu > 0 is STABLE (attractor)")

## Existence and Uniqueness Theory

### Python Implementation: Checking Existence and Uniqueness

In [ ]:
import numpy as np
import sympy as sp

def check_picard_lindelof(f_expr, f_prime_expr, t0, y0, a=2.0, b=2.0):
    """
    Numerically probe the Picard-Lindelof hypotheses for dy/dt = f(t, y)
    on the rectangle |t - t0| <= a, |y - y0| <= b.
    """
    t, y = sp.symbols('t y')
    f_lambda = sp.lambdify((t, y), f_expr, 'numpy')
    df_dy_lambda = sp.lambdify((t, y), f_prime_expr, 'numpy')

    # M = max |f| on the (sampled) rectangle
    t_vals = np.linspace(t0 - a, t0 + a, 50)
    y_vals = np.linspace(y0 - b, y0 + b, 50)
    T, Y = np.meshgrid(t_vals, y_vals)
    F = np.abs(f_lambda(T, Y) * np.ones_like(T))
    M = np.max(F)

    # Guaranteed existence interval
    h = min(a, b / (M + 1e-10))

    print("Picard-Lindelof Analysis:")
    print(f"  Rectangle: t in [{t0-a}, {t0+a}], y in [{y0-b}, {y0+b}]")
    print(f"  Max |f| on rectangle: M = {M:.4f}")
    print(f"  Guaranteed uniqueness interval: |t - {t0}| <= {h:.4f}")

    # Probe the Lipschitz constant on two grid resolutions. If the
    # estimate keeps growing under refinement, |df/dy| is likely
    # unbounded on R, and the Lipschitz condition may fail.
    L_estimates = []
    for n in (50, 1000):
        tg = np.linspace(t0 - a, t0 + a, n)
        yg = np.linspace(y0 - b, y0 + b, n)
        Tg, Yg = np.meshgrid(tg, yg)
        with np.errstate(divide='ignore', invalid='ignore'):
            vals = np.abs(df_dy_lambda(Tg, Yg) * np.ones_like(Tg))
        L_estimates.append(np.max(vals[np.isfinite(vals)]))
    L_coarse, L_fine = L_estimates

    print(f"  Max |df/dy|: {L_coarse:.4f} (coarse grid), "
          f"{L_fine:.4f} (fine grid)")
    if L_fine <= 1.5 * L_coarse:
        print("  [OK] df/dy appears bounded: Lipschitz condition holds")
    else:
        print("  [WARN] |df/dy| grows under grid refinement: "
              "Lipschitz condition may fail")

    return h, M, L_fine

# Example 1: Well-behaved ODE (uniqueness guaranteed)
t, y = sp.symbols('t y')
f1 = -y          # dy/dt = -y
f1_prime = sp.Integer(-1)
h1, M1, L1 = check_picard_lindelof(f1, f1_prime, 0, 1)

print("\n" + "="*60 + "\n")

# Example 2: Non-Lipschitz ODE (potential non-uniqueness at y = 0)
f2 = sp.sqrt(sp.Abs(y))   # dy/dt = sqrt(|y|)
f2_prime = sp.sign(y) / (2*sp.sqrt(sp.Abs(y)))
h2, M2, L2 = check_picard_lindelof(f2, f2_prime, 0, 0.1)

## Autonomous Equations and Phase Line Analysis

### Python Implementation: Automated Phase Line Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import sympy as sp

class PhaseLineAnalyzer:
    """Automated phase line analysis for dy/dt = f(y)"""

    def __init__(self, f_sympy, y_sym):
        """
        f_sympy: SymPy expression for f(y)
        y_sym: SymPy symbol for y
        """
        self.f_expr = f_sympy
        self.y_sym = y_sym
        self.f_lambda = sp.lambdify(y_sym, f_sympy, 'numpy')
        self.df_expr = sp.diff(f_sympy, y_sym)
        self.df_lambda = sp.lambdify(y_sym, self.df_expr, 'numpy')

    def find_equilibria(self, y_range, n_samples=1000):
        """Find equilibrium points in given y range."""
        y_vals = np.linspace(y_range[0], y_range[1], n_samples)
        f_vals = self.f_lambda(y_vals)

        # Find sign changes
        sign_changes = np.where(np.diff(np.sign(f_vals)))[0]
        equilibria = []

        for idx in sign_changes:
            try:
                eq = brentq(self.f_lambda, y_vals[idx], y_vals[idx+1])
                equilibria.append(eq)
            except ValueError:
                pass

        return np.array(equilibria)

    def classify_equilibria(self, equilibria):
        """Classify equilibria as stable, unstable, or semi-stable."""
        classification = {}
        for eq in equilibria:
            deriv = self.df_lambda(eq)
            if deriv < -1e-6:
                classification[eq] = 'stable (sink)'
            elif deriv > 1e-6:
                classification[eq] = 'unstable (source)'
            else:
                classification[eq] = 'semi-stable (center)'
        return classification

    def plot_phase_line(self, y_range, figsize=(12, 4)):
        """Create comprehensive phase line diagram."""
        y_vals = np.linspace(y_range[0], y_range[1], 1000)
        f_vals = self.f_lambda(y_vals)

        # Find equilibria
        equilibria = self.find_equilibria(y_range)
        classification = self.classify_equilibria(equilibria)

        fig, axes = plt.subplots(1, 2, figsize=figsize)

        # Plot 1: f(y) vs y
        ax = axes[0]
        ax.plot(y_vals, f_vals, 'b-', linewidth=2.5, label='f(y)')
        ax.axhline(0, color='k', linestyle='-', alpha=0.3)
        ax.axvline(0, color='k', linestyle='-', alpha=0.3)

        # Mark equilibria
        colors = {'stable (sink)': 'green', 'unstable (source)': 'red',
                 'semi-stable (center)': 'orange'}
        for eq, class_type in classification.items():
            ax.plot(eq, 0, 'o', markersize=10,
                   color=colors.get(class_type, 'gray'))

        ax.set_xlabel('y', fontsize=11)
        ax.set_ylabel('f(y)', fontsize=11)
        ax.set_title('Direction Field: dy/dt = f(y)', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=10)

        # Plot 2: Phase line
        ax = axes[1]
        ax.set_ylim(-0.5, 0.5)
        ax.set_xlim(y_range[0], y_range[1])

        # Draw y-axis (phase line)
        ax.axhline(0, color='k', linewidth=2)

        # Add direction arrows
        arrow_y_pos = 0
        for y in np.linspace(y_range[0], y_range[1], 20):
            if abs(self.f_lambda(y)) > 1e-6:
                direction = 1 if self.f_lambda(y) > 0 else -1
                ax.arrow(y, arrow_y_pos, direction*0.4, 0,
                        head_width=0.08, head_length=0.1,
                        fc='blue', ec='blue', alpha=0.6)

        # Mark equilibria
        for eq, class_type in classification.items():
            marker = 'o' if 'stable' in class_type else 'x'
            color = colors.get(class_type, 'gray')
            size = 12 if 'stable' in class_type else 15
            ax.plot(eq, 0, marker, markersize=size, color=color,
                   markeredgewidth=2, label=class_type)

        ax.set_xlabel('y', fontsize=11)
        ax.set_title('Phase Line Diagram', fontsize=12)
        ax.set_yticks([])
        ax.grid(True, alpha=0.3, axis='x')
        ax.legend(loc='upper right', fontsize=9)

        plt.tight_layout()
        plt.show()

        # Print summary
        print("Phase Line Analysis Summary")
        print("="*50)
        for eq in equilibria:
            print(f"  y* = {eq:.6f}: {classification[eq]}")

        return equilibria, classification

# Example: Analyze dy/dt = y^3 - y
y = sp.symbols('y', real=True)
f = y**3 - y
analyzer = PhaseLineAnalyzer(f, y)
equilibria, classification = analyzer.plot_phase_line((-2, 2))

## Applications in Fluid Mechanics and Heat Transfer

### Python Implementation: Engineering Applications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============ Hagen-Poiseuille Flow ============
def poiseuille_flow(r_array, R=1.0, dp_dx=-1.0, mu=0.001):
    """
    Compute Hagen-Poiseuille velocity profile.
    v_x(r) = -(1/(4*mu)) * (dp/dx) * (R^2 - r^2)
    """
    v_x = -(1.0 / (4.0 * mu)) * dp_dx * (R**2 - r_array**2)
    return v_x

# Parameters
R_pipe = 0.01  # meters
L_pipe = 1.0   # meters
mu_fluid = 0.001  # Pa*s
delta_P = 100  # Pa (pressure drop)
dp_dx = -delta_P / L_pipe

# Compute velocity profile
r_vals = np.linspace(0, R_pipe, 100)
v_profile = poiseuille_flow(r_vals, R=R_pipe, dp_dx=dp_dx, mu=mu_fluid)
v_max = poiseuille_flow(0, R=R_pipe, dp_dx=dp_dx, mu=mu_fluid)

# Volume flow rate
Q = -(np.pi * R_pipe**4 / (8.0 * mu_fluid)) * dp_dx

print("Hagen-Poiseuille Flow Analysis")
print(f"  Pipe radius:      {R_pipe*1000:.1f} mm")
print(f"  Pressure drop:    {delta_P:.1f} Pa")
print(f"  Maximum velocity: {v_max:.4f} m/s")
print(f"  Volume flow rate: {Q*1e6:.2f} cm^3/s\n")

# ============ Fin Heat Transfer ============
def fin_temperature(x_array, T_b=100, T_inf=20, L=0.1, m=5.0):
    """
    Fin temperature distribution using hyperbolic functions.
    """
    T_vals = T_inf + (T_b - T_inf) * np.sinh(m * (L - x_array)) / np.sinh(m * L)
    return T_vals

def fin_heat_rate(T_b, T_inf, L, m, h, P, k, A_c):
    """
    Total heat convected from the fin surface (tip held at ambient).
    Q_fin = sqrt(h*P*k*A_c)*(T_b-T_inf) * (cosh(mL)-1)/sinh(mL)
    """
    cosh_mL = np.cosh(m * L)
    sinh_mL = np.sinh(m * L)
    M_fin = np.sqrt(h * P * k * A_c) * (T_b - T_inf)
    return M_fin * (cosh_mL - 1.0) / sinh_mL

# Fin parameters
T_base = 100  # deg C
T_ambient = 20
L_fin = 0.05  # m
h_conv = 100  # W/(m^2*K)
P_perim = 0.1  # m (perimeter)
k_thermal = 50  # W/(m*K)
A_cross = 0.001  # m^2
m_param = np.sqrt(h_conv * P_perim / (k_thermal * A_cross))

x_fin = np.linspace(0, L_fin, 100)
T_fin = fin_temperature(x_fin, T_b=T_base, T_inf=T_ambient,
                        L=L_fin, m=m_param)
Q_fin = fin_heat_rate(T_base, T_ambient, L_fin, m_param, h_conv,
                      P_perim, k_thermal, A_cross)

print("Fin Heat Transfer Analysis")
print(f"  Fin length:           {L_fin*100:.1f} cm")
print(f"  Base temperature:     {T_base:.1f} C")
print(f"  Ambient temperature:  {T_ambient:.1f} C")
print(f"  Heat transfer rate:   {Q_fin:.2f} W")

# ============ Plotting ============
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Velocity profile
ax = axes[0]
ax.plot(r_vals*1000, v_profile, 'b-', linewidth=2.5)
ax.fill_between(r_vals*1000, 0, v_profile, alpha=0.3)
ax.set_xlabel('Radial Distance (mm)', fontsize=11)
ax.set_ylabel('Velocity v_x (m/s)', fontsize=11)
ax.set_title('Hagen-Poiseuille Velocity Profile', fontsize=12)
ax.grid(True, alpha=0.3)
ax.axvline(R_pipe*1000, color='r', linestyle='--', alpha=0.5)
ax.text(R_pipe*1000+0.2, v_max*0.8, 'Wall', fontsize=10)

# Temperature profile
ax = axes[1]
ax.plot(x_fin*100, T_fin, 'r-', linewidth=2.5)
ax.fill_between(x_fin*100, T_ambient, T_fin, alpha=0.3, color='red')
ax.axhline(T_ambient, color='b', linestyle='--', label='Ambient')
ax.axhline(T_base, color='k', linestyle='--', label='Base')
ax.set_xlabel('Position Along Fin (cm)', fontsize=11)
ax.set_ylabel('Temperature (C)', fontsize=11)
ax.set_title('Fin Temperature Distribution', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()